In [ ]:
import json
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../dadosDesafio/dados_nivel_1.json")

with DATA_PATH.open(encoding="utf-8") as file:
    dataset = json.load(file)

df = pd.DataFrame(dataset["operacoes"])

taxa_usd_brl = dataset["taxa_cambio_usd_brl"]

print(f"Taxa USD/BRL: {taxa_usd_brl}")
print(f"Quantidade de operações: {len(df)}")


df.head()

df.info()

df.describe(include="all")

print("Valores ausentes:")
display(df.isna().sum())

duplicados = df[df.duplicated(subset="id", keep=False)].sort_values("id")

display(duplicados)

df = df.drop_duplicates(subset="id", keep="first").copy()

df["data"] = pd.to_datetime(
    df["data"],
    errors="coerce"
)

print(f"Operações após deduplicação: {len(df)}"),
print(f"IDs únicos: {df['id'].nunique()}")
print(f"Datas ausentes: {df['data'].isna().sum()}")

df["valor_brl"] = df["valor"]

mask_usd = df["moeda"].eq("USD")

df.loc[mask_usd, "valor_brl"] = (
    df.loc[mask_usd, "valor"] * taxa_usd_brl
)

Taxa USD/BRL: 5.4
Quantidade de operações: 20
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.5 KB
Valores ausentes:


id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


Operações após deduplicação: 19
IDs únicos: 19
Datas ausentes: 1


TypeError: Invalid value '[64800.]' for dtype 'int64'

# Desafio Itaú Estágio Engenharia de IA — Nível 1
## Prevenção à Lavagem de Dinheiro

Este notebook apresenta o tratamento dos dados, a aplicação de regras
determinísticas e a análise interpretativa de um cliente sinalizado
utilizando um modelo de linguagem.

A solução mantém separadas as responsabilidades:

- pandas: limpeza, agregações e cálculos determinísticos;
- regras: identificação objetiva dos sinais definidos no desafio;
- LLM: interpretação dos sinais e redação do parecer;
- validação estruturada: garantia de que a resposta da LLM segue o formato esperado.

## Tratamento dos problemas de qualidade

Foram identificados dois problemas relevantes:

1. O identificador `OP-0007` aparece duas vezes com os mesmos valores
   em todos os campos. A segunda ocorrência foi considerada uma duplicação
   do registro e apenas uma ocorrência foi mantida.

2. A operação `OP-0017` possui data ausente. A data não foi inferida ou
   preenchida artificialmente, pois não existe evidência suficiente para
   determinar o dia correto. Essa operação permanece na base para as
   análises que não dependem de data, mas não participa da regra de
   fracionamento, que exige operações realizadas na mesma data.

A deduplicação é importante porque uma duplicação alteraria contagens,
medianas e agregações. Já a preservação da operação sem data evita a
introdução de uma informação que não está presente na fonte.